# GRU：更简洁的门控循环单元
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：GRU.py | 核心功能：双门控机制、参数更少、效果接近 LSTM

## 概述

GRU（Gated Recurrent Unit）是 LSTM 的简化版，用两个门（重置门和更新门）替代三个门，合并了细胞状态和隐状态。参数量约为 LSTM 的 75%，在很多任务上效果相当。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

## 数学原理

### 1. GRU 的门控机制

**代码对应**：GRU 是 LSTM 的简化版本，合并了细胞状态和隐藏状态。

**重置门**：$\mathbf{r}_t = \sigma(\mathbf{W}_r[\mathbf{h}_{t-1}, \mathbf{x}_t])$

**更新门**：$\mathbf{z}_t = \sigma(\mathbf{W}_z[\mathbf{h}_{t-1}, \mathbf{x}_t])$

**候选隐藏状态**：$\tilde{\mathbf{h}}_t = \tanh(\mathbf{W}_h[\mathbf{r}_t \odot \mathbf{h}_{t-1}, \mathbf{x}_t])$

**隐藏状态更新**：$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$

### 2. GRU vs LSTM

| 特性 | LSTM | GRU |
|------|------|-----|
| 门数 | 3（遗忘、输入、输出） | 2（重置、更新） |
| 状态 | $\mathbf{h}_t$ 和 $\mathbf{c}_t$ 分离 | 只有 $\mathbf{h}_t$ |
| 参数量 | $4(d_h^2 + d_h d_x + d_h)$ | $3(d_h^2 + d_h d_x + d_h)$ |
| 训练速度 | 较慢 | 较快（参数少 ~25%） |
| 长序列 | 通常更好 | 类似或稍差 |

GRU 的更新门 $\mathbf{z}_t$ 同时扮演 LSTM 的遗忘门和输入门的角色：当 $\mathbf{z}_t \approx 0$ 时保留旧状态，当 $\mathbf{z}_t \approx 1$ 时用新候选替换。

### 1. 序列数据生成

运行 1. 序列数据生成 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
timesteps = 100

def generate_sine_data(n_samples, timesteps):
    X, y = [], []
    for _ in range(n_samples):
        start = np.random.uniform(0, 2 * np.pi)
        t = np.linspace(start, start + 4 * np.pi, timesteps + 1)
# --- 数值计算 ---
        data = np.sin(t) + 0.1 * np.random.randn(len(t))
        X.append(data[:-1])
        y.append(data[-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = generate_sine_data(500, timesteps)
X_train, X_test = torch.FloatTensor(X[:400]).unsqueeze(-1), torch.FloatTensor(X[400:]).unsqueeze(-1)
y_train, y_test = torch.FloatTensor(y[:400]), torch.FloatTensor(y[400:])

### 2. GRU 模型

运行 2. GRU 模型 的代码，观察算法在该环节的行为。

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1, dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
# --- 继续 ---
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, hn = self.gru(x, h0)
        out = self.fc(out[:, -1, :])
        return out.squeeze(-1)

model = GRUModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"=== GRU 模型结构 ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

### 3. GRU 门控机制

运行 3. GRU 门控机制 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== GRU 门控机制 ===")
print("更新门 z_t = σ(W_z × [h_{t-1}, x_t]) — 决定保留多少旧状态")
print("重置门 r_t = σ(W_r × [h_{t-1}, x_t]) — 决定遗忘多少旧状态来计算候选值")
print("候选值 h̃_t = tanh(W × [r_t × h_{t-1}, x_t])")
print("新状态 h_t = (1 - z_t) × h_{t-1} + z_t × h̃_t")
# --- 输出结果 ---
print()
print("与 LSTM 的区别:")
print("  GRU 有 2 个门（更新门、重置门），LSTM 有 3 个门（遗忘门、输入门、输出门）")
print("  GRU 没有独立的细胞状态，参数量约为 LSTM 的 3/4")

### 4. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练 ===")
n_epochs = 50
for epoch in range(n_epochs):
    model.train()
    output = model(X_train)
# --- 继续 ---
    loss = criterion(output, y_train)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_rmse = torch.sqrt(criterion(model(X_test), y_test)).item()
        print(f"  Epoch {epoch+1:>2}: Train RMSE={torch.sqrt(loss).item():.4f}, "
# --- 赋值/配置 ---
              f"Test RMSE={test_rmse:.4f}")

### 5. GRU vs LSTM vs RNN 对比

对比不同模型或配置的性能差异。

In [ ]:
from torch.nn import RNN as SimpleRNN

class SimpleRNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = SimpleRNN(1, 64, 2, batch_first=True)
        self.fc = nn.Linear(64, 1)
# --- 定义函数 ---
    def forward(self, x):
        out, _ = self.rnn(x, torch.zeros(2, x.size(0), 64))
        return self.fc(out[:, -1, :]).squeeze(-1)

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 64, 2, batch_first=True)
        self.fc = nn.Linear(64, 1)
# --- 定义函数 ---
    def forward(self, x):
        out, _ = self.lstm(x, (torch.zeros(2, x.size(0), 64), torch.zeros(2, x.size(0), 64)))
        return self.fc(out[:, -1, :]).squeeze(-1)

print("\n=== RNN vs GRU vs LSTM 对比 ===")
for name, Model in [("RNN", SimpleRNNModel), ("GRU", GRUModel), ("LSTM", LSTMModel)]:
    m = Model()
    n_params = sum(p.numel() for p in m.parameters())
    opt = optim.Adam(m.parameters(), lr=0.001)
# --- 循环处理 ---
    for _ in range(50):
        m.train()
        loss = criterion(m(X_train), y_train)
        opt.zero_grad()
        loss.backward()
# --- 继续 ---
        torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
        opt.step()
    m.eval()
    with torch.no_grad():
        rmse = torch.sqrt(criterion(m(X_test), y_test)).item()
# --- 输出结果 ---
    print(f"  {name}: 参数量={n_params:>6}, Test RMSE={rmse:.4f}")

print("\n=== GRU 要点 ===")
print("- 参数比 LSTM 少，训练更快")
print("- 效果通常与 LSTM 相当（有时更好，有时稍差）")
print("- 适合：数据量较小或需要快速实验时")
print("- 与 LSTM 一样可捕捉长距离依赖")

## 关键代码解释

```python
gru = nn.GRU(input_size=10, hidden_size=20, num_layers=2, batch_first=True)
output, hn = gru(x)  # 没有单独的细胞状态
```

更新门 z_t 控制保留多少旧信息、接收多少新信息。重置门 r_t 控制如何结合旧隐状态和新输入。

## 注意事项

1. **训练更快**：参数比 LSTM 少
2. **选择建议**：先试 GRU，不够再换 LSTM
3. **小数据集上差异不大**

## 总结与延伸

以上代码展示了 **GRU** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **LSTM vs GRU**：没有绝对优劣，取决于任务和数据
- **SRU（Simple Recurrent Unit）**：进一步简化，可并行化
- **State Space Models**：Mamba 等新架构正在取代 RNN
